# 🗂️ Notebook 2: Reminder / Alert — Data Model & APIs

Here we pin down **what data we store** and **how clients talk to us**.
These two choices shape every component downstream, so we spend the
effort to get the invariants right.

You'll build a tiny, fully in-memory `ReminderService` — no DB, no HTTP
server — but the shape of it is exactly what a production service
exposes. By the end you'll be able to `schedule`, `cancel`, `get`, and
watch a reminder get delivered (and deduped on retry).


## 🛠️ Setup

```bash
cd 06-system-designs/reminder-alert
uv sync
```

Then in VS Code: pick the `.venv` kernel (top-right of the notebook).
If it doesn't show up, run `Cmd+Shift+P` → **Reload Window**.

All code here uses only the Python standard library + `pydantic`. No
servers, no databases — everything runs in-process so you can step
through the ideas.


## 🧱 Entities

Two is enough to start:

- **`Reminder`** — *what* to fire, *when*, *to whom*, *on which channel*.
  Its `status` walks a little state machine:
  `scheduled → firing → done` (or → `cancelled`, or → `failed`).
- **`Delivery`** — one *attempt* to deliver a reminder. We record every
  attempt so we can retry intelligently and audit failures later.

Separating "what should happen" from "what actually happened" is a
classic modelling win: the user's intent (Reminder) is immutable-ish,
while the delivery log grows as we retry.


In [ ]:
# Pydantic v2 gives us validation, serialization, and clear type errors —
# all without writing boilerplate __init__ methods.
from __future__ import annotations
from datetime import datetime, timezone
from typing import Literal, Optional
from pydantic import BaseModel, Field, field_validator

Channel = Literal["push", "email", "sms"]
ReminderStatus = Literal["scheduled", "firing", "done", "cancelled", "failed"]
DeliveryStatus = Literal["ok", "retry", "dead"]


class Reminder(BaseModel):
    id: str
    user_id: str
    # We *always* store the fire time in UTC. The user's local timezone
    # is stored separately so we can redisplay it correctly.
    fire_at: datetime
    tz: str = "UTC"                       # e.g. "America/New_York"
    channel: Channel
    payload: str = Field(max_length=500)
    # Optional simple recurrence. For complex patterns you'd use RRULE.
    recurrence: Optional[Literal["daily", "weekly", "monthly"]] = None
    status: ReminderStatus = "scheduled"
    # Idempotency key: the API uses this to dedupe POST retries from
    # flaky mobile clients.
    idempotency_key: Optional[str] = None

    @field_validator("fire_at")
    @classmethod
    def _must_be_utc(cls, v: datetime) -> datetime:
        if v.tzinfo is None:
            raise ValueError("fire_at must be timezone-aware")
        return v.astimezone(timezone.utc)


class Delivery(BaseModel):
    reminder_id: str
    attempt: int                          # 1, 2, 3, ...
    status: DeliveryStatus
    ts: datetime
    error: Optional[str] = None

# Smoke test
r = Reminder(
    id="r1", user_id="u1",
    fire_at=datetime(2030, 1, 1, 8, 0, tzinfo=timezone.utc),
    channel="push", payload="Take your pills 💊",
)
print(r.model_dump_json(indent=2))


### Why these choices?

- **UTC everywhere** in storage. The golden rule of distributed
  scheduling: convert to UTC at the API edge, store UTC, convert back
  to local only when rendering. This avoids DST bugs where "8 AM" moves
  an hour twice a year.
- **Separate `tz` field** lets us *recompute* the next fire time for
  recurring reminders (see below).
- **`idempotency_key`** is how we stop duplicate reminders when a mobile
  app retries a flaky POST. Stripe-style keys, nothing exotic.
- **`max_length=500`** on payload: surprisingly effective at blocking
  "accidentally paste a 10 MB blob" bugs.


## 🌐 Timezones and DST done right

`datetime` without a timezone is a landmine. Python 3.9+ ships
`zoneinfo` which reads the system tz database — we use it for both
"convert local → UTC" and for recurrence arithmetic.


In [ ]:
from zoneinfo import ZoneInfo
from datetime import datetime, timezone

def to_utc(local_wall_clock: datetime, tz: str) -> datetime:
    """Interpret a naive datetime as wall-clock time in `tz`, return UTC."""
    assert local_wall_clock.tzinfo is None, "pass a naive datetime"
    return local_wall_clock.replace(tzinfo=ZoneInfo(tz)).astimezone(timezone.utc)

# DST in the US "springs forward" at 02:00 local on the second Sunday
# of March — in 2025 that's March 9. So the 8 AM local slot on
# Saturday is in EST (-05:00) and on Sunday it's in EDT (-04:00),
# putting the *next* 8 AM only 23 UTC-hours after the previous one.
sat_8am = to_utc(datetime(2025, 3, 8, 8, 0), "America/New_York")
sun_8am = to_utc(datetime(2025, 3, 9, 8, 0), "America/New_York")
mon_8am = to_utc(datetime(2025, 3, 10, 8, 0), "America/New_York")

print("Sat 8 AM NY :", sat_8am.isoformat(), "UTC")
print("Sun 8 AM NY :", sun_8am.isoformat(), "UTC")
print("Mon 8 AM NY :", mon_8am.isoformat(), "UTC")
print("Sat→Sun gap :", (sun_8am - sat_8am))   # 23 hours — DST spring-forward
print("Sun→Mon gap :", (mon_8am - sun_8am))   # back to 24 hours


Notice the **23-hour gap** on the DST boundary (Sat → Sun). This is
why recurring reminders can't just add 24 hours to the previous fire
time — you must do the arithmetic in the *user's* timezone, then
convert to UTC.


## 🔌 HTTP APIs (the contract)

| Method | Path                     | What |
|--------|--------------------------|------|
| `POST`   | `/reminders`             | Schedule (body: `user_id, fire_at, tz, channel, payload, recurrence?`) |
| `GET`    | `/reminders/{id}`        | Details + delivery log |
| `PATCH`  | `/reminders/{id}`        | Reschedule (body: new `fire_at`) |
| `DELETE` | `/reminders/{id}`        | Cancel |
| `GET`    | `/users/{uid}/reminders` | List (paginated) |

Clients send an **`Idempotency-Key`** header on POSTs. If we see the same
key twice, we return the original resource instead of creating a
duplicate.


## 🧪 Runnable: an in-memory `ReminderService`

Same shape as a real service — just uses a dict instead of a database.
Notice the boundaries: `schedule_reminder` validates + dedupes, a
`tick()` method plays the role of the scheduler loop.


In [ ]:
import uuid
from datetime import datetime, timezone, timedelta
from typing import Optional
from zoneinfo import ZoneInfo
import calendar


class ReminderService:
    def __init__(self):
        self._reminders: dict[str, Reminder] = {}
        self._deliveries: list[Delivery] = []
        # Maps idempotency_key -> reminder_id so retries are free.
        self._idem: dict[str, str] = {}

    # ----- write API -----
    def schedule_reminder(
        self,
        user_id: str,
        fire_at: datetime,
        channel: Channel,
        payload: str,
        tz: str = "UTC",
        recurrence: Optional[str] = None,
        idempotency_key: Optional[str] = None,
    ) -> Reminder:
        if idempotency_key and idempotency_key in self._idem:
            # Retry with same key → return the already-created reminder.
            return self._reminders[self._idem[idempotency_key]]

        reminder = Reminder(
            id=str(uuid.uuid4()),
            user_id=user_id,
            fire_at=fire_at,
            tz=tz,
            channel=channel,
            payload=payload,
            recurrence=recurrence,
            idempotency_key=idempotency_key,
        )
        self._reminders[reminder.id] = reminder
        if idempotency_key:
            self._idem[idempotency_key] = reminder.id
        return reminder

    def cancel(self, reminder_id: str) -> bool:
        r = self._reminders.get(reminder_id)
        if not r or r.status != "scheduled":
            return False
        self._reminders[reminder_id] = r.model_copy(update={"status": "cancelled"})
        return True

    def reschedule(self, reminder_id: str, new_fire_at: datetime) -> bool:
        r = self._reminders.get(reminder_id)
        if not r or r.status != "scheduled":
            return False
        self._reminders[reminder_id] = r.model_copy(update={"fire_at": new_fire_at})
        return True

    # ----- read API -----
    def get(self, reminder_id: str) -> Optional[Reminder]:
        return self._reminders.get(reminder_id)

    def deliveries_for(self, reminder_id: str) -> list[Delivery]:
        return [d for d in self._deliveries if d.reminder_id == reminder_id]

    # ----- scheduler loop (simplified) -----
    def tick(self, now: Optional[datetime] = None) -> list[Reminder]:
        """Fire every due reminder and return what was fired."""
        now = now or datetime.now(timezone.utc)
        fired: list[Reminder] = []
        for r in list(self._reminders.values()):
            if r.status == "scheduled" and r.fire_at <= now:
                self._deliver(r)
                fired.append(r)
        return fired

    # ----- internal -----
    def _deliver(self, r: Reminder) -> None:
        self._deliveries.append(Delivery(
            reminder_id=r.id, attempt=1, status="ok",
            ts=datetime.now(timezone.utc),
        ))
        if r.recurrence:
            next_fire = _advance(r.fire_at, r.tz, r.recurrence)
            self._reminders[r.id] = r.model_copy(update={"fire_at": next_fire})
        else:
            self._reminders[r.id] = r.model_copy(update={"status": "done"})


def _advance(fire_at_utc: datetime, tz: str, pattern: str) -> datetime:
    """Advance `fire_at` by one recurrence step, respecting `tz` (DST-safe)."""
    local = fire_at_utc.astimezone(ZoneInfo(tz))
    if pattern == "daily":
        local = local + timedelta(days=1)
    elif pattern == "weekly":
        local = local + timedelta(days=7)
    elif pattern == "monthly":
        # "The 31st of every month" does not exist in February. The naive
        # `replace(month=2)` on Jan 31 raises ValueError and takes the whole
        # scheduler down at 08:00 on Feb 1. Clamp to the last valid day, which
        # is what iCalendar and every calendar app do.
        month = local.month + 1
        year = local.year + (month - 1) // 12
        month = (month - 1) % 12 + 1
        day = min(local.day, calendar.monthrange(year, month)[1])
        local = local.replace(year=year, month=month, day=day)
    else:
        raise ValueError(f"unknown recurrence {pattern!r}")
    return local.astimezone(timezone.utc)

print("ReminderService ready.")

In [ ]:
# Demo: schedule, idempotent retry, list, fire, see delivery log.
svc = ReminderService()

now = datetime.now(timezone.utc)

r1 = svc.schedule_reminder(
    user_id="alice",
    fire_at=now - timedelta(seconds=1),       # already due
    channel="push",
    payload="Take your pills 💊",
    tz="America/New_York",
    idempotency_key="mobile-req-42",
)
# Same idempotency key → we get r1 back, NOT a new row.
r1_again = svc.schedule_reminder(
    user_id="alice", fire_at=now - timedelta(seconds=1),
    channel="push", payload="Take your pills 💊",
    tz="America/New_York", idempotency_key="mobile-req-42",
)
assert r1.id == r1_again.id, "idempotency failed!"

r2 = svc.schedule_reminder(
    user_id="bob",
    fire_at=now + timedelta(hours=1),         # future
    channel="email",
    payload="Weekly report ready",
    tz="UTC",
)

fired = svc.tick()
print(f"Fired {len(fired)} reminder(s):")
for r in fired:
    print(f"  {r.id[:8]}  user={r.user_id}  channel={r.channel}  → {r.payload}")

print("\nDelivery log for r1:")
for d in svc.deliveries_for(r1.id):
    print(f"  attempt={d.attempt}  status={d.status}  at {d.ts.isoformat()}")

print(f"\nr2 still pending? {svc.get(r2.id).status == 'scheduled'}")


In [ ]:
# Recurring reminder demo: daily at 8 AM New York, even across DST.
svc2 = ReminderService()

# Interpret "2025-03-08 08:00 NY" and store as UTC.
first_fire = to_utc(datetime(2025, 3, 8, 8, 0), "America/New_York")
r = svc2.schedule_reminder(
    user_id="carla", fire_at=first_fire, channel="push",
    payload="☀️ Daily standup", tz="America/New_York", recurrence="daily",
)

# Fast-forward 3 days by repeatedly ticking with a fake "now".
for day_offset in range(1, 4):
    fake_now = first_fire + timedelta(days=day_offset, hours=1)
    svc2.tick(now=fake_now)
    next_fire_local = svc2.get(r.id).fire_at.astimezone(ZoneInfo("America/New_York"))
    print(f"after day {day_offset}: next fires at {next_fire_local.isoformat()}")


### Two calendar traps, tested

Recurrence arithmetic is where reminder systems get their 3 a.m. pages. Both of
these are one-liners to get wrong and impossible to notice until the date rolls
around.

In [ ]:
# Trap 1 — DST. A "daily" reminder must stay at the same LOCAL wall-clock time,
# which means the UTC gap between consecutive fires is not always 24 hours.
sat = to_utc(datetime(2025, 3, 8, 8, 0), "America/New_York")
sun = _advance(sat, "America/New_York", "daily")
assert (sun - sat) == timedelta(hours=23), "spring-forward day is 23 UTC hours"
assert sun.astimezone(ZoneInfo("America/New_York")).hour == 8, "still 8 AM locally"

nov1 = to_utc(datetime(2025, 11, 1, 8, 0), "America/New_York")
nov2 = _advance(nov1, "America/New_York", "daily")
assert (nov2 - nov1) == timedelta(hours=25), "fall-back day is 25 UTC hours"
print("✅ DST: daily advance is 23h / 24h / 25h in UTC, always 8 AM locally")

# Trap 2 — short months. Jan 31 + 1 month is not a date.
jan31 = to_utc(datetime(2025, 1, 31, 9, 0), "UTC")
feb   = _advance(jan31, "UTC", "monthly")
assert feb.day == 28, feb          # clamped, not crashed
mar   = _advance(feb, "UTC", "monthly")
print(f"✅ monthly: Jan 31 → {feb:%b %d} → {mar:%b %d}")
print("   ⚠️  Note the drift: once we clamp to Feb 28 we never get back to the")
print("   31st. A correct implementation stores the ORIGINAL day-of-month on the")
print("   reminder and re-derives each occurrence from it, instead of chaining")
print("   off the previous fire time. That is why iCalendar RRULE keeps a")
print("   DTSTART: recurrence must be a function of the anchor, not of the last run.")

## 🧠 Takeaways

- Store UTC, remember the user's timezone, do recurrence arithmetic in
  the **local** zone — then convert back.
- `Reminder` (intent) and `Delivery` (attempt) are two tables because
  they change at very different rates.
- The **idempotency key** on POST is how you survive flaky mobile
  retries without creating 3 copies of the same reminder.
- A `tick()` method you can call with a **fake `now`** is a superpower
  for testing time-based systems — no `time.sleep` in tests.

Next up: the real **scheduling algorithm** — how to pick what fires next
without scanning a billion rows.
